# GABM Evaluation Benchmark Notebook

This notebook benchmarks the paper's **baseline SFM evacuation model** against the **behavioral-grid (GABM) model** across multiple random seeds.

**Questions answered**
1. Is the behavioral model consistently faster in early phase?
2. Does it increase congestion near the exit?
3. Are improvements robust or seed-specific?



In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

def find_repo_path(repo_name='FYP-Generative-Agent-Evacuation-Simulation'):
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        candidate = p / repo_name
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find '{repo_name}' from: {start}")

repo_root = find_repo_path()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pysocialforce as psf
from pysocialforce.sceneforevacuation import PedStateEvacuation, EnvStateEvacuation

np.random.seed(42)
print('Using repo:', repo_root)
print('pysocialforce from:', psf.__file__)


Using repo: C:\Users\kylet\HTX\Deliverable 1\FYP-Generative-Agent-Evacuation-Simulation
pysocialforce from: C:\Users\kylet\HTX\Deliverable 1\FYP-Generative-Agent-Evacuation-Simulation\pysocialforce\__init__.py


## Experimental Design

- Same room geometry, same exits, same agent count for both models.
- Same random seed per pair (baseline vs behavioral).
- Metrics:
  - `total_time_s`: total evacuation duration
  - `t25_s`: time to 25% evacuated (early-phase speed)
  - `congestion_index`: mean active agents near exit over time

Interpretation:
- Lower `t25_s` means better early throughput.
- Lower `congestion_index` means less local crowding near the exit.



In [2]:
def create_square_room_with_exit(size=20.0, door_width=1.2):
    h = door_width / 2
    return [
        [-size/2, size/2, -size/2, -size/2],
        [-size/2, size/2,  size/2,  size/2],
        [-size/2, -size/2, -size/2, size/2],
        [ size/2,  size/2, -size/2, -h],
        [ size/2,  size/2,  h, size/2],
    ]

def generate_positions(rng, n_agents, room_size=20.0, margin=2.0):
    return rng.uniform(-room_size/2 + margin, room_size/2 - margin, size=(n_agents, 2))

def generate_characteristics(rng, n_agents):
    age_categories = ['Young Adult', 'Adult', 'Middle Age']
    genders = ['male', 'female']
    trait_levels = ['Low', 'High']
    chars = []
    for _ in range(n_agents):
        age = rng.choice(age_categories)
        gender = rng.choice(genders)
        traits = [rng.choice(trait_levels) for _ in range(5)]
        chars.append([age, gender] + traits)
    return chars

def build_initial_states(seed, n_agents=50, room_size=20.0, tau=0.5):
    rng = np.random.default_rng(seed)
    door_center = np.array([room_size / 2, 0.0])
    final_goal = door_center + np.array([3.0, 0.0])

    pos0 = generate_positions(rng, n_agents, room_size)
    vel0 = np.zeros((n_agents, 2), dtype=float)
    for i in range(n_agents):
        d = final_goal - pos0[i]
        d = d / np.linalg.norm(d)
        vel0[i] = d * 0.5

    goals = np.tile(final_goal, (n_agents, 1))
    chars = generate_characteristics(rng, n_agents)

    # Baseline [n, 6]
    baseline_state = np.zeros((n_agents, 6), dtype=float)
    baseline_state[:, :2] = pos0
    baseline_state[:, 2:4] = vel0
    baseline_state[:, 4:6] = goals

    # Behavioral [n, 14]
    behavioral_state = np.empty((n_agents, 14), dtype=object)
    behavioral_state[:, :2] = pos0
    behavioral_state[:, 2:4] = vel0
    behavioral_state[:, 4:6] = goals
    behavioral_state[:, 6] = tau
    for i, c in enumerate(chars):
        behavioral_state[i, 7:] = c

    return baseline_state, behavioral_state



In [3]:
def time_to_fraction(escaped_counts, times, n_agents, frac=0.25):
    target = int(np.ceil(n_agents * frac))
    idx = np.where(np.array(escaped_counts) >= target)[0]
    return float(times[idx[0]]) if len(idx) else np.nan

def run_one(state, behavioral, obstacles, cfg, behavior_file=None, max_steps=3000, tau=0.5, exit_x=10.0):
    if behavioral:
        sim = psf.SimulatorWithBehavior(
            state,
            groups=None,
            obstacles=obstacles,
            config_file=cfg,
            behavior_file=behavior_file,
        )
    else:
        sim = psf.Simulator(
            state,
            groups=None,
            obstacles=obstacles,
            config_file=cfg,
            ped_state_class=PedStateEvacuation,
            env_state_class=EnvStateEvacuation,
        )

    n_agents = state.shape[0]
    escaped_counts = [0]
    times = [0.0]
    near_exit_counts = []

    for step in range(1, max_steps + 1):
        sim.step(1)
        escaped = int(np.sum(sim.peds.escaped))
        escaped_counts.append(escaped)
        times.append(step * tau)

        pos = sim.peds.pos()
        active = ~sim.peds.escaped
        near_exit = (np.abs(pos[:, 0] - exit_x) <= 1.0) & (np.abs(pos[:, 1]) <= 1.5) & active
        near_exit_counts.append(int(np.sum(near_exit)))

        if escaped == n_agents:
            break

    return {
        'total_time_s': float(times[-1]),
        't25_s': time_to_fraction(escaped_counts, times, n_agents, frac=0.25),
        'congestion_index': float(np.mean(near_exit_counts)) if near_exit_counts else np.nan,
        'escaped_final': int(escaped_counts[-1]),
        'n_agents': int(n_agents),
    }

def run_pair(seed, n_agents=50, max_steps=3000, tau=0.5):
    obstacles = create_square_room_with_exit(size=20.0, door_width=1.2)
    cfg = repo_root / 'experiments' / 'example.toml'
    behavior_file = repo_root / 'behavior grid analysis' / 'behavior impatience analysis' / 'behavior_grid.json'

    baseline_state, behavioral_state = build_initial_states(seed=seed, n_agents=n_agents, tau=tau)

    base = run_one(baseline_state, behavioral=False, obstacles=obstacles, cfg=cfg, max_steps=max_steps, tau=tau)
    behv = run_one(behavioral_state, behavioral=True, obstacles=obstacles, cfg=cfg, behavior_file=behavior_file, max_steps=max_steps, tau=tau)

    row = {
        'seed': seed,
        'base_total_time_s': base['total_time_s'],
        'behv_total_time_s': behv['total_time_s'],
        'base_t25_s': base['t25_s'],
        'behv_t25_s': behv['t25_s'],
        'base_congestion': base['congestion_index'],
        'behv_congestion': behv['congestion_index'],
    }
    row['delta_t25_pct'] = ((row['base_t25_s'] - row['behv_t25_s']) / row['behv_t25_s'] * 100.0) if row['behv_t25_s'] > 0 else np.nan
    row['delta_congestion_pct'] = ((row['behv_congestion'] - row['base_congestion']) / row['base_congestion'] * 100.0) if row['base_congestion'] > 0 else np.nan
    return row



In [ ]:
# Run benchmark (adjust N_RUNS up if you have time)
N_RUNS = 8
SEEDS = list(range(100, 100 + N_RUNS))

rows = []
for s in SEEDS:
    rows.append(run_pair(seed=s, n_agents=50, max_steps=3000, tau=0.5))

df = pd.DataFrame(rows)
summary = df.describe().T[['mean', 'std', 'min', 'max']]

print('Per-seed results:')
display(df)
print('\nSummary statistics:')
display(summary)


DEBUG:[byteflow.py:81             __init__() ] bytecode dump:
>          0	NOP(arg=None, lineno=97)
           2	RESUME(arg=0, lineno=97)
           4	LOAD_FAST(arg=0, lineno=101)
           6	LOAD_CONST(arg=1, lineno=101)
           8	LOAD_CONST(arg=1, lineno=101)
          10	BUILD_SLICE(arg=2, lineno=101)
          12	LOAD_CONST(arg=2, lineno=101)
          14	LOAD_CONST(arg=3, lineno=101)
          16	BUILD_SLICE(arg=2, lineno=101)
          18	BUILD_TUPLE(arg=2, lineno=101)
          20	BINARY_SUBSCR(arg=None, lineno=101)
          24	STORE_FAST(arg=1, lineno=101)
          26	LOAD_GLOBAL(arg=1, lineno=102)
          36	LOAD_ATTR(arg=2, lineno=102)
          56	LOAD_FAST(arg=1, lineno=102)
          58	GET_ITER(arg=None, lineno=102)
          60	LOAD_FAST_AND_CLEAR(arg=2, lineno=102)
          62	SWAP(arg=2, lineno=102)
          64	BUILD_LIST(arg=0, lineno=102)
          66	SWAP(arg=2, lineno=102)
>         68	FOR_ITER(arg=33, lineno=102)
          72	STORE_FAST(arg=2, lineno=102

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].boxplot([df['base_t25_s'].dropna(), df['behv_t25_s'].dropna()], labels=['Baseline', 'Behavioral'])
axes[0].set_title('T25 (lower is better)')
axes[0].set_ylabel('Seconds')
axes[0].grid(True, axis='y', alpha=0.3)

axes[1].boxplot([df['base_congestion'].dropna(), df['behv_congestion'].dropna()], labels=['Baseline', 'Behavioral'])
axes[1].set_title('Congestion Index near exit (lower is better)')
axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()



## Findings Template (Fill after run)

- Early phase: Behavioral model is ___ than baseline on `T25` by about ___% on average.
- Congestion: Behavioral model shows ___ congestion near exits by about ___% on average.
- Robustness: Across ___ seeds, the trend is (consistent / mixed / unstable).

### Recommendation for supervisor
Use GABM for richer behavior stress-tests, but keep baseline SFM as control and require multi-seed reporting + calibration before operational claims.



**Research-Informed Model Comparison (Paper + External)**

| Approach | Strength | Limitation | Best Use |
|---|---|---|---|
| Baseline SFM | Interpretable physics; established in evacuation literature | Limited psychology/heterogeneity | Control baseline, quick scenario checks |
| SFM + Behavioral Grid (this project) | Captures trait-conditioned urgency and heterogeneity | Depends on prompt design + calibration assumptions | Research prototypes and what-if behavior studies |
| Extended SFM with emotion/personality calibration | Can better align with observed panic heterogeneity | Needs richer data and validation effort | High-fidelity study where survey/drill data exists |
| Cellular Automata variants | Efficient for large populations | Coarser movement realism | Large-scale stress screening |

Literature pointers are listed in the final markdown cell below.



**References**

1. Helbing, D., & Molnar, P. (1995). Social force model for pedestrian dynamics. *Phys. Rev. E*. DOI: 10.1103/PhysRevE.51.4282
2. Helbing, D., Farkas, I., & Vicsek, T. (2000). Simulating dynamical features of escape panic. *Nature*. DOI: 10.1038/35035023
3. Park, J. S., et al. (2023). Generative Agents: Interactive Simulacra of Human Behavior. arXiv:2304.03442
4. Horton, J. J., Filippas, A., & Manning, B. S. (2023, rev. 2026). Large Language Models as Simulated Economic Agents. NBER Working Paper 31122.
5. Wang, Y., et al. (2021). A novel emergency evacuation model considering personality traits. *Sustainability*, 13(18), 10463.
6. Wang, X., et al. (2023). Heterogeneous crowd dynamics considering personality traits under fire emergency. *Physica A*, 610, 128411.
7. Zhang, Y., et al. (2023). Modified social force model considering emotional contagion for crowd evacuation simulation. *IJDRR*, 96, 103902.

